# NOAA Severe Weather Events

## Data needed:
- Zillow County level data with FIPS
- NOAA event details with FIPS

## Data cleaning
- Damage to crops and damage to property need to be cleaned of NA and put into numeric values
- deaths and injuries need to be cleaned of NA


## Steps
- Load Zillow data 
- Load NOAA files and merge together fatalities, details
- Flooding - boolean based on flood_cause
- Tornado - boolean based on tor_f_scale


## Analytical metrics
For each event id
- performance of the regional zillow index at months T+1, T+2, T+3, T+4, T+5, T+6
- performance of the national zillow index at months T+1, T+2, T+3, T+4, T+5, T+6
- damage to crops 
- damage to property
- flooding boolean
- tornado boolean
- duration of event END - BEGIN
- breadth of event RANGE 

In [1]:
import pandas as pd

In [2]:
noaa_2025_fatalities = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_fatalities-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2025_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260116.csv.gz"
noaa_2025_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2025_c20260116.csv.gz"

In [3]:
details = pd.read_csv(noaa_2025_details)
fatalities = pd.read_csv(noaa_2025_fatalities)
locations = pd.read_csv(noaa_2025_locations)

In [4]:
df = pd.merge(details, locations, on="EVENT_ID", how="left")

In [5]:
df.columns

Index(['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH',
       'END_DAY', 'END_TIME', 'EPISODE_ID_x', 'EVENT_ID', 'STATE',
       'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS',
       'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME',
       'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT',
       'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE',
       'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE',
       'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE',
       'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE',
       'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH',
       'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
       'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'YEARMONTH',
       'EPISODE_ID_y', 'LOCATION_INDEX', 'RANGE', 'AZIMUTH', 'LOCATION',
       'LATITUDE', 'LONGITUDE', 'LAT2', 'LON2'],
      dtype='objec

Parse the crop and property damage fields into numeric values

In [6]:
def parse_numeric_string(rawstring):
    if pd.isna(rawstring):
        return 0
    if "K" in rawstring:
        value = rawstring.replace("K","")
        value = float(value) * 1000
        return value
    if "M" in rawstring:
        value = rawstring.replace("M","")
        value = float(value) * 1000000
        return value
    if "B" in rawstring:
        value = rawstring.replace("B","")
        value = float(value) * 1000000000
        return value
    return float(rawstring)
df['DAMAGE_PROPERTY'] = df["DAMAGE_PROPERTY"].apply(parse_numeric_string)
df['DAMAGE_CROPS'] = df["DAMAGE_CROPS"].apply(parse_numeric_string)

In [7]:
def has_flooding(rawstring):
    if pd.isna(rawstring):
        return 0
    return 1

df["FLOODING"] = df["FLOOD_CAUSE"].apply(has_flooding)
df["FLOODING"]

0        0
1        0
2        0
3        0
4        0
        ..
84957    0
84958    1
84959    1
84960    1
84961    1
Name: FLOODING, Length: 84962, dtype: int64

In [8]:
def has_tornado(rawstring):
    if pd.isna(rawstring):
        return 0
    if len(rawstring) > 0:
        return 1

df["TORNADO"] = df["TOR_F_SCALE"].apply(has_tornado)
df["TORNADO"]

0        0
1        1
2        0
3        0
4        0
        ..
84957    0
84958    0
84959    0
84960    0
84961    0
Name: TORNADO, Length: 84962, dtype: int64

In [11]:

df["BEGIN_DATE_TIME"] = pd.to_datetime(df["BEGIN_DATE_TIME"])
df["END_DATE_TIME"] = pd.to_datetime(df["END_DATE_TIME"])
duration_delta = df["END_DATE_TIME"] - df["BEGIN_DATE_TIME"]
df["Duration_Hours"] = duration_delta.dt.total_seconds() / 3600.0
df["Duration_Hours"]

0         0.033333
1         0.050000
2        28.450000
3         6.000000
4         6.000000
           ...    
84957     0.000000
84958     1.533333
84959     1.533333
84960     1.533333
84961     1.533333
Name: Duration_Hours, Length: 84962, dtype: float64

In [10]:

df.loc[:, "RANGE"] 

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
         ...  
84957    12.15
84958     7.12
84959     7.05
84960     2.49
84961     3.32
Name: RANGE, Length: 84962, dtype: float64